In [2]:
import gymnasium as gym
from gymnasium.spaces import Discrete
import numpy as np
import random
from typing import Optional, Any

In [ ]:
class DiscreteStateDiscreteAction(gym.Env):
    """Пример окружения с дискретными состояниями и действиями"""

    metadata = {"render_modes": ["human"], "render_fps": 4}

    def __init__(
        self,
        *,
        render_mode: Optional[str] = None,
        seed: Optional[int] = None,
        max_steps: int = 10
    ) -> None:
        super().__init__()

        # Дискретное пространство состояний и действий
        self.observation_space = Discrete(3)  # 0, 1, 2 → вверх / без изменений / вниз
        self.action_space = Discrete(3)       # 0, 1, 2 → продать / держать / купить

        # Семантические описания (удобно для отладки)
        self.observation_in_words = ['вверх', 'без изменений', 'вниз']
        self.action_in_words = ['продать', 'держать', 'купить']

        # Состояние среды
        self.state = 0
        self.step_count = 0

        # Параметры визуализации
        self.render_mode = render_mode

        # Инициализация случайности
        self.np_random, _ = gym.utils.seeding.np_random(seed)
        
        self.max_steps = max_steps

    # -----------------------
    # Основные методы Gym API
    # -----------------------

    def reset(
        self, 
        *, 
        seed: Optional[int] = None, 
        options: Optional[dict[str, Any]] = None
    ):
        super().reset(seed=seed)
        self.state = self.np_random.integers(0, 3)
        self.step_count = 0
        info = {}
        
        if self.render_mode == "human":
            print(f"[reset] Начальное состояние: {self.observation_in_words[self.state]}")
        
        return self.state, info

    def step(self, action: int):
        # Проверка допустимости действия
        assert self.action_space.contains(action), f"Недопустимое действие: {action}"

        # Простейшая динамика: новое состояние выбирается случайно
        new_state = self.np_random.integers(0, 3)

        # Вознаграждение: +1, если направление действия совпадает с направлением рынка
        reward = 1.0 if action == new_state else -0.5

        self.state = new_state
        self.step_count += 1

        # Прекращение эпизода через 10 шагов
        terminated = self.step_count >= self.max_steps
        truncated = False  # Можно добавить ограничение по времени
        info = {}

        if self.render_mode == "human":
            print(f"[step {self.step_count}] Действие: {self.action_in_words[action]} | "
                  f"Состояние: {self.observation_in_words[self.state]} | Награда: {reward}")

        return self.state, reward, terminated, truncated, info

    def render(self):
        print(f"Текущее состояние: {self.observation_in_words[self.state]}")

    def close(self):
        pass

In [6]:
env = DiscreteStateDiscreteAction(render_mode="human")
obs, info = env.reset()

done = False
while not done:
    action = env.action_space.sample()  # случайное действие
    obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated

env.close()

[reset] Начальное состояние: вниз
[step 1] Действие: продать | Состояние: вверх | Награда: 1.0
[step 2] Действие: купить | Состояние: вниз | Награда: 1.0
[step 3] Действие: продать | Состояние: без изменений | Награда: -0.5
[step 4] Действие: держать | Состояние: без изменений | Награда: 1.0
[step 5] Действие: держать | Состояние: вверх | Награда: -0.5
[step 6] Действие: держать | Состояние: вниз | Награда: -0.5
[step 7] Действие: купить | Состояние: вверх | Награда: -0.5
[step 8] Действие: купить | Состояние: вверх | Награда: -0.5
[step 9] Действие: держать | Состояние: вниз | Награда: -0.5
[step 10] Действие: купить | Состояние: вверх | Награда: -0.5
